# Microbiome Alpha Diversity Analysis
## Shannon Diversity Over Time with Vitamin D Intervention

This notebook analyzes alpha diversity changes between two groups across 14 timepoints.

## 1. Setup

In [ ]:
setwd('/data/home-Documents/1-Grants/Gill/Results/')

library(ggplot2)
library(dplyr)
library(phyloseq)
library(ggpubr)
library(effects)
library(broom)
library(randomcoloR)

set.seed(100)

# Load pre-built phyloseq object (or build from DADA2 pipeline)
load('phyloseq_vitD.RData')

## 2. Outlier Detection and Removal

In [ ]:
alpha_div_check <- estimate_richness(ps, measures = "Shannon")
metadata_check <- as(sample_data(ps), "data.frame")
alpha_div_check <- cbind(alpha_div_check, metadata_check)

# Fit initial model to identify outliers
lm_check <- lm(Shannon ~ Time * Group, data = alpha_div_check)
std_resid <- rstandard(lm_check)
outliers_idx <- which(abs(std_resid) > 3)
outlier_samples <- rownames(alpha_div_check)[outliers_idx]

cat("Outliers identified:", length(outlier_samples), "\n")
if (length(outlier_samples) > 0) {
  print(alpha_div_check[outliers_idx, c("SampleID", "Time", "Group", "Shannon")])
}

In [ ]:
# Create cleaned phyloseq object
samples_to_keep <- sample_names(ps)[!sample_names(ps) %in% outlier_samples]
ps_clean <- prune_samples(samples_to_keep, ps)
pslog_clean <- transform_sample_counts(ps_clean, function(x) log(1 + x))

cat("Original samples:", nsamples(ps), "\n")
cat("After outlier removal:", nsamples(ps_clean), "\n")

## 3. Calculate Alpha Diversity

In [ ]:
alpha_div <- estimate_richness(ps_clean, measures = c("Observed", "Shannon", "Simpson"))
metadata <- as(sample_data(ps_clean), "data.frame")
alpha_div <- cbind(alpha_div, metadata)

# Order Time factor correctly
alpha_div$WeekNum <- as.numeric(gsub("Week", "", alpha_div$Time))
alpha_div$Time <- factor(alpha_div$Time, 
                         levels = unique(alpha_div$Time[order(alpha_div$WeekNum)]))

head(alpha_div)

In [ ]:
# Summary statistics
summary_df <- alpha_div %>%
  group_by(Time, Group) %>%
  summarise(
    n = n(),
    mean_shannon = mean(Shannon, na.rm = TRUE),
    sd_shannon = sd(Shannon, na.rm = TRUE),
    se_shannon = sd(Shannon, na.rm = TRUE) / sqrt(n()),
    .groups = "drop"
  )

summary_df$WeekNum <- as.numeric(gsub("Week", "", summary_df$Time))
summary_df$Time <- factor(summary_df$Time, 
                          levels = unique(summary_df$Time[order(summary_df$WeekNum)]))
print(summary_df)

## 4. Visualizations

In [ ]:
# Line plot
ggplot(summary_df, aes(x = Time, y = mean_shannon, group = Group, color = Group)) +
  geom_line(linewidth = 1.2) +
  geom_point(size = 3) +
  geom_errorbar(aes(ymin = mean_shannon - se_shannon, ymax = mean_shannon + se_shannon), width = 0.2) +
  theme_minimal() +
  labs(title = "Shannon Diversity Over Time", y = "Mean Shannon Index ± SE", x = "Time") +
  scale_color_manual(values = c("G1" = "red", "G2" = "blue")) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

In [ ]:
# Boxplot
ggplot(alpha_div, aes(x = Time, y = Shannon, fill = Group)) +
  geom_boxplot(outlier.shape = NA, alpha = 0.7, position = position_dodge(0.8)) +
  geom_jitter(aes(color = Group), size = 1.5, alpha = 0.6,
              position = position_jitterdodge(jitter.width = 0.2, dodge.width = 0.8)) +
  theme_minimal() +
  labs(title = "Shannon Diversity by Group and Time", y = "Shannon Index", x = "Time") +
  scale_fill_manual(values = c("G1" = "red", "G2" = "blue")) +
  scale_color_manual(values = c("G1" = "red", "G2" = "blue")) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

## 5. Per-Week Statistical Tests

In [ ]:
safe_wilcox <- function(data) {
  tryCatch({
    if (length(unique(data$Group)) < 2 || sum(data$Group == "G1") < 2 || sum(data$Group == "G2") < 2) return(NA)
    wilcox.test(Shannon ~ Group, data = data)$p.value
  }, error = function(e) NA)
}

week_stats <- alpha_div %>%
  group_by(Time) %>%
  summarise(n_G1 = sum(Group == "G1"), n_G2 = sum(Group == "G2"),
            p_value = safe_wilcox(cur_data()), .groups = "drop") %>%
  mutate(p_adj = p.adjust(p_value, method = "fdr"),
         significance = case_when(is.na(p_adj) ~ "NA", p_adj < 0.001 ~ "***",
                                   p_adj < 0.01 ~ "**", p_adj < 0.05 ~ "*", TRUE ~ "NS"))

print(week_stats)

## 6. Two-Way ANOVA

In [ ]:
lm_clean <- lm(Shannon ~ Time * Group, data = alpha_div)

cat("========== ANOVA Results ==========\n")
print(anova(lm_clean))

In [ ]:
cat("========== Model Summary ==========\n")
print(summary(lm_clean))

In [ ]:
# Diagnostic plots
par(mfrow = c(2, 2))
plot(lm_clean)

## 7. Effects Plot

In [ ]:
eff_data <- as.data.frame(Effect(c("Time", "Group"), lm_clean))
eff_data <- left_join(eff_data, week_stats %>% select(Time, significance), by = "Time")

ggplot(eff_data, aes(x = Time, y = fit, group = Group, color = Group, fill = Group)) +
  geom_line(linewidth = 1.2) +
  geom_ribbon(aes(ymin = lower, ymax = upper), alpha = 0.2, color = NA) +
  geom_point(size = 2) +
  scale_color_manual(values = c("G1" = "red", "G2" = "blue")) +
  scale_fill_manual(values = c("G1" = "red", "G2" = "blue")) +
  labs(title = "Predicted Shannon Diversity by Group and Time",
       subtitle = paste("Time:Group interaction p =", format(anova(lm_clean)["Time:Group", "Pr(>F)"], digits = 3)),
       x = "Time", y = "Predicted Shannon Index") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

## 8. Taxonomic Composition Barplots

In [ ]:
# Color palette for taxa

taxa_colors <- distinctColorPalette(n_distinct(species_data$Species))

In [ ]:
# Species-level barplot (>5% relative abundance)

# --- Species-level: identify species with prevalence 1 ---
species_all <- pslog_clean %>%
  tax_glom(taxrank = "Species") %>%
  transform_sample_counts(function(x) x / sum(x)) %>%
  psmelt()

species_stats <- species_all %>%
  group_by(Species, Group) %>%
  summarise(
    mean_abund = mean(Abundance),
    max_abund = max(Abundance),
    n_mice_above_5pct = sum(Abundance > 0.05),
    total_mice = n(),
    prevalence = n_mice_above_5pct / total_mice,
    .groups = "drop"
  )

one_mouse_genera <- species_stats %>%
  filter(n_mice_above_5pct == 1, max_abund > 0.10) %>%
  pull(Species)

cat("One-mouse-wonder genera removed:", length(one_mouse_genera), "\n")
print(one_mouse_genera)

# --- Remove from phyloseq object ---
tt <- as.data.frame(tax_table(pslog_clean))
taxa_to_remove <- rownames(tt)[tt$Species %in% one_mouse_genera]
taxa_to_keep <- setdiff(taxa_names(pslog_clean), taxa_to_remove)
pslog_filtered <- prune_taxa(taxa_to_keep, pslog_clean)

# --- Species-level barplot (>5% relative abundance) ---
species_data <- pslog_filtered %>%
  tax_glom(taxrank = "Species") %>%
  transform_sample_counts(function(x) x / sum(x)) %>%
  psmelt() %>%
  filter(Abundance > 0.05) %>%
  arrange(Species)

species_data$WeekNum <- as.numeric(gsub("Week", "", species_data$Time))
species_data$Time <- factor(species_data$Time, 
                          levels = unique(species_data$Time[order(species_data$WeekNum)]))

cat("Number of genera with >5% abundance:", n_distinct(species_data$Species), "\n")

# --- Generate enough colors ---
taxa_colors <- colorRampPalette(RColorBrewer::brewer.pal(12, "Paired"))(n_distinct(species_data$Species))

# --- Plot ---
ggplot(species_data, aes(x = Time, y = Abundance, fill = Species)) +
  facet_grid(Group ~ .) +
  geom_bar(stat = "identity") +
  scale_fill_manual(values = taxa_colors) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = "Species Composition Over Time", 
       y = "Relative Abundance (Genera > 5%)", x = "")

In [ ]:
# Genus-level barplot (>5% relative abundance)

# --- Genus-level: identify species with prevalence 1 ---
genus_all <- pslog_clean %>%
  tax_glom(taxrank = "Genus") %>%
  transform_sample_counts(function(x) x / sum(x)) %>%
  psmelt()

genus_stats <- genus_all %>%
  group_by(Genus, Group) %>%
  summarise(
    mean_abund = mean(Abundance),
    max_abund = max(Abundance),
    n_mice_above_5pct = sum(Abundance > 0.05),
    total_mice = n(),
    prevalence = n_mice_above_5pct / total_mice,
    .groups = "drop"
  )

one_mouse_genera <- genus_stats %>%
  filter(n_mice_above_5pct == 1, max_abund > 0.10) %>%
  pull(Genus)

cat("One-mouse-wonder genera removed:", length(one_mouse_genera), "\n")
print(one_mouse_genera)

# --- Remove from phyloseq object ---
tt <- as.data.frame(tax_table(pslog_clean))
taxa_to_remove <- rownames(tt)[tt$Genus %in% one_mouse_genera]
taxa_to_keep <- setdiff(taxa_names(pslog_clean), taxa_to_remove)
pslog_filtered <- prune_taxa(taxa_to_keep, pslog_clean)

# --- Genus-level barplot (>5% relative abundance) ---
genus_data <- pslog_filtered %>%
  tax_glom(taxrank = "Genus") %>%
  transform_sample_counts(function(x) x / sum(x)) %>%
  psmelt() %>%
  filter(Abundance > 0.05) %>%
  arrange(Genus)

genus_data$WeekNum <- as.numeric(gsub("Week", "", genus_data$Time))
genus_data$Time <- factor(genus_data$Time, 
                          levels = unique(genus_data$Time[order(genus_data$WeekNum)]))

cat("Number of genera with >5% abundance:", n_distinct(genus_data$Genus), "\n")

# --- Generate enough colors ---
taxa_colors <- colorRampPalette(RColorBrewer::brewer.pal(12, "Paired"))(n_distinct(genus_data$Genus))

# --- Plot ---
ggplot(genus_data, aes(x = Time, y = Abundance, fill = Genus)) +
  facet_grid(Group ~ .) +
  geom_bar(stat = "identity") +
  scale_fill_manual(values = taxa_colors) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = "Genus Composition Over Time", 
       y = "Relative Abundance (Genera > 5%)", x = "")

## 9. PERMANOVA analysis

In [ ]:
library(vegan) library(phyloseq)

===== SPECIES LEVEL COMPARISON =====

ps_species <- tax_glom(pslog, taxrank = "Species")
Get unique weeks

weeks <- unique(sample_data(ps_species)$Time)
Store results

species_results <- list()

for(week in weeks) {
Subset to just this week

ps_week <- subset_samples(ps_species, Time == week)
Calculate distance

dist_week <- phyloseq::distance(ps_week, method = "bray")
Get metadata

metadata_week <- as(sample_data(ps_week), "data.frame")
Run PERMANOVA comparing groups at this week

permanova_week <- adonis2(dist_week ~ Group, data = metadata_week, permutations = 999)

species_results[[week]] <- permanova_week

cat("\n========================================\n") cat("SPECIES LEVEL - ", week, "\n") cat("========================================\n") 
print(permanova_week) }

===== GENUS LEVEL COMPARISON =====

ps_genus <- tax_glom(pslog, taxrank = "Genus")

genus_results <- list()

for(week in weeks) {
Subset to just this week

ps_week <- subset_samples(ps_genus, Time == week)
Calculate distance

dist_week <- phyloseq::distance(ps_week, method = "bray")
Get metadata

metadata_week <- as(sample_data(ps_week), "data.frame")
Run PERMANOVA comparing groups at this week

permanova_week <- adonis2(dist_week ~ Group, data = metadata_week, permutations = 999)

genus_results[[week]] <- permanova_week

cat("\n========================================\n") cat("GENUS LEVEL - ", week, "\n") cat("========================================\n")
print(permanova_week) }

## 10. Create a summary table of p-values from PERMANOVA

In [ ]:
# Extract p-values and R² values into a nice summary table

# Function to extract results
extract_permanova_results <- function(results_list, level_name) {
  summary_df <- data.frame(
    Week = names(results_list),
    Taxonomic_Level = level_name,
    R_squared = numeric(length(results_list)),
    P_value = numeric(length(results_list)),
    Significant = character(length(results_list)),
    stringsAsFactors = FALSE
  )
  
  for(i in seq_along(results_list)) {
    result <- results_list[[i]]
    summary_df$R_squared[i] <- result$R2[1]
    summary_df$P_value[i] <- result$`Pr(>F)`[1]
    summary_df$Significant[i] <- ifelse(result$`Pr(>F)`[1] < 0.05, "Yes", "No")
  }
  
  return(summary_df)
}

# Create summary tables
species_summary <- extract_permanova_results(species_results, "Species")
genus_summary <- extract_permanova_results(genus_results, "Genus")

# Combine
combined_summary <- rbind(species_summary, genus_summary)

print(combined_summary)

# Save to CSV
write.csv(combined_summary, "PERMANOVA_Group_Comparisons_by_Week.csv", row.names = FALSE)

# 11. Complete DESeq2 Analysis by Week

In [ ]:

> # ===== RUN ANALYSIS FOR ALL WEEKS =====
> 
> # Get unique weeks
> weeks <- unique(as.character(sample_data(ps)$Time))
> print("Available weeks:")
[1] "Available weeks:"
> print(weeks)
 [1] "Week0"  "Week6"  "Week7"  "Week8"  "Week9"  "Week10" "Week11" "Week1"  "Week12" "Week13" "Week2"  "Week3"  "Week4"  "Week5" 
> 
> # Check unique groups
> groups <- unique(as.character(sample_data(ps)$Group))
> print("Available groups:")
[1] "Available groups:"
> print(groups)
[1] "G1" "G2"
> 
> # Store all results
> all_species_results <- list()
> all_genus_results <- list()
> 
> # Run for each week
> for(week in weeks) {
+   cat("\n==============================================\n")
+   cat("Processing:", week, "\n")
+   cat("==============================================\n")
+   
+   # Species level
+   species_res <- run_deseq_by_week(ps, week, tax_level = "Species")
+   
+   if(!is.null(species_res)) {
+     all_species_results[[week]] <- species_res
+     
+     # Print significant results
+     sig_species <- species_res %>% filter(padj < 0.05)
+     cat("\nSPECIES - Significant taxa (padj < 0.05):", nrow(sig_species), "\n")
+     
+     if(nrow(sig_species) > 0) {
+       print(sig_species %>% select(Species, log2FoldChange, padj) %>% head(10))
+     }
+   }
+   
+   # Genus level
+   genus_res <- run_deseq_by_week(ps, week, tax_level = "Genus")
+   
+   if(!is.null(genus_res)) {
+     all_genus_results[[week]] <- genus_res
+     
+     # Print significant results
+     sig_genus <- genus_res %>% filter(padj < 0.05)
+     cat("\nGENUS - Significant taxa (padj < 0.05):", nrow(sig_genus), "\n")
+     
+     if(nrow(sig_genus) > 0) {
+       print(sig_genus %>% select(Genus, log2FoldChange, padj) %>% head(10))
+     }
+   }
+ }

==============================================
Processing: Week0 
==============================================
Error in eval(e, x, parent.frame()) : object 'week_name' not found
Called from: eval(e, x, parent.frame())
```

# 12. Summarize significant taxa per week

In [ ]:
# ===== SUMMARY OF SIGNIFICANT TAXA ONLY =====

library(dplyr)
library(knitr)

# Function to create clean summary for significant taxa
create_significant_summary <- function(results_list, tax_level = "Species") {
  
  significant_list <- list()
  
  for(week in names(results_list)) {
    # Get significant taxa for this week
    sig_taxa <- results_list[[week]] %>%
      filter(padj < 0.05) %>%
      arrange(padj) %>%
      select(all_of(tax_level), log2FoldChange, padj, baseMean)
    
    if(nrow(sig_taxa) > 0) {
      sig_taxa$Week <- week
      sig_taxa$Direction <- ifelse(sig_taxa$log2FoldChange > 0, 
                                     "Enriched in G2", 
                                     "Enriched in G1")
      significant_list[[week]] <- sig_taxa
    }
  }
  
  if(length(significant_list) > 0) {
    combined <- bind_rows(significant_list)
    return(combined)
  } else {
    return(NULL)
  }
}

# Create summaries
cat("\n==============================================\n")
cat("SUMMARY OF SIGNIFICANT SPECIES\n")
cat("==============================================\n")

sig_species_summary <- create_significant_summary(all_species_results, "Species")

if(!is.null(sig_species_summary)) {
  # Reorder columns
  sig_species_summary <- sig_species_summary %>%
    select(Week, Species, Direction, log2FoldChange, padj, baseMean) %>%
    arrange(Week, padj)
  
  print(sig_species_summary)
  write.csv(sig_species_summary, "Significant_Species_Summary.csv", row.names = FALSE)
  cat("\nSaved to: Significant_Species_Summary.csv\n")
} else {
  cat("No significant species found.\n")
}


cat("\n==============================================\n")
cat("SUMMARY OF SIGNIFICANT GENERA\n")
cat("==============================================\n")

sig_genus_summary <- create_significant_summary(all_genus_results, "Genus")

if(!is.null(sig_genus_summary)) {
  # Reorder columns
  sig_genus_summary <- sig_genus_summary %>%
    select(Week, Genus, Direction, log2FoldChange, padj, baseMean) %>%
    arrange(Week, padj)
  
  print(sig_genus_summary)
  write.csv(sig_genus_summary, "Significant_Genus_Summary.csv", row.names = FALSE)
  cat("\nSaved to: Significant_Genus_Summary.csv\n")
} else {
  cat("No significant genera found.\n")
}


# ===== COUNT SUMMARY TABLE =====

cat("\n==============================================\n")
cat("COUNT SUMMARY BY WEEK\n")
cat("==============================================\n")

count_summary <- data.frame(
  Week = character(),
  Level = character(),
  Total_Significant = integer(),
  Enriched_G1 = integer(),
  Enriched_G2 = integer(),
  stringsAsFactors = FALSE
)

# Add species counts
if(!is.null(sig_species_summary)) {
  species_counts <- sig_species_summary %>%
    group_by(Week) %>%
    summarise(
      Level = "Species",
      Total_Significant = n(),
      Enriched_G1 = sum(Direction == "Enriched in G1"),
      Enriched_G2 = sum(Direction == "Enriched in G2")
    )
  count_summary <- bind_rows(count_summary, species_counts)
}

# Add genus counts
if(!is.null(sig_genus_summary)) {
  genus_counts <- sig_genus_summary %>%
    group_by(Week) %>%
    summarise(
      Level = "Genus",
      Total_Significant = n(),
      Enriched_G1 = sum(Direction == "Enriched in G1"),
      Enriched_G2 = sum(Direction == "Enriched in G2")
    )
  count_summary <- bind_rows(count_summary, genus_counts)
}

print(count_summary)
write.csv(count_summary, "Significant_Taxa_Counts.csv", row.names = FALSE)


# ===== TOP 5 MOST SIGNIFICANT PER WEEK =====

cat("\n==============================================\n")
cat("TOP 5 MOST SIGNIFICANT SPECIES PER WEEK\n")
cat("==============================================\n")

if(!is.null(sig_species_summary)) {
  top5_species <- sig_species_summary %>%
    group_by(Week) %>%
    slice_min(order_by = padj, n = 5) %>%
    select(Week, Species, Direction, log2FoldChange, padj)
  
  print(top5_species)
  write.csv(top5_species, "Top5_Species_PerWeek.csv", row.names = FALSE)
}


cat("\n==============================================\n")
cat("TOP 5 MOST SIGNIFICANT GENERA PER WEEK\n")
cat("==============================================\n")

if(!is.null(sig_genus_summary)) {
  top5_genus <- sig_genus_summary %>%
    group_by(Week) %>%
    slice_min(order_by = padj, n = 5) %>%
    select(Week, Genus, Direction, log2FoldChange, padj)
  
  print(top5_genus)
  write.csv(top5_genus, "Top5_Genus_PerWeek.csv", row.names = FALSE)
}


# ===== NICE FORMATTED TABLE FOR REPORT =====

cat("\n==============================================\n")
cat("FORMATTED TABLE FOR REPORTING\n")
cat("==============================================\n")

if(!is.null(sig_species_summary)) {
  formatted_table <- sig_species_summary %>%
    mutate(
      log2FC = round(log2FoldChange, 2),
      `p-adj` = format(padj, scientific = TRUE, digits = 2),
      Abundance = round(baseMean, 0)
    ) %>%
    select(Week, Species, Direction, log2FC, `p-adj`, Abundance) %>%
    arrange(Week, `p-adj`)
  
  print(kable(formatted_table, format = "simple"))
  write.csv(formatted_table, "Formatted_Significant_Species.csv", row.names = FALSE)
}

## 13. Save Results

In [ ]:
saveRDS(ps_clean, "ps_clean.rds")
write.csv(week_stats, "alpha_diversity_week_stats.csv", row.names = FALSE)
write.csv(summary_df, "alpha_diversity_summary.csv", row.names = FALSE)
write.csv(broom::tidy(anova(lm_clean)), "anova_results.csv", row.names = FALSE)

cat("Analysis complete. Results saved.\n")